# Q3 – Text Summarization
## CENG 467 – Natural Language Understanding and Generation
### Take-Home Midterm

| | |
|---|---|
| **Dataset** | CNN/DailyMail (subset of 500 test articles) |
| **Extractive** | TextRank (graph-based sentence scoring via `sumy`) |
| **Abstractive** | BART-large-cnn (pre-trained, inference only) |
| **Metrics** | ROUGE-1/2/L · BLEU · METEOR · BERTScore |
| **Seed** | 42 |


In [ ]:
!pip install datasets "transformers<5.0.0" torch sumy rouge-score bert-score nltk \
             sacrebleu matplotlib -q
import nltk
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet",   quiet=True)
nltk.download("omw-1.4",   quiet=True)
print("Done.")


In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print("Imports OK.")


## §1 – Dataset: CNN/DailyMail

CNN/DailyMail contains ~300 K English news articles paired with human-written
highlights (multi-sentence abstractive summaries):

| Property | Value |
|---|---|
| Total articles | ~312 K |
| Subset used | 500 test articles (reproducible via seed=42) |
| Avg article length | ~780 words |
| Avg summary length | ~56 words |
| Summary style | Bullet-point highlights (abstractive) |

We use the `3.0.0` version (standard benchmark for summarisation).


In [ ]:
from datasets import load_dataset

raw_cnn = load_dataset("cnn_dailymail", "3.0.0")
# Use test split – subsample 500 articles for speed
rng = np.random.default_rng(SEED)
test_indices = rng.choice(len(raw_cnn["test"]), size=500, replace=False).tolist()
test_subset  = raw_cnn["test"].select(test_indices)

articles  = test_subset["article"]
summaries = test_subset["highlights"]    # reference summaries

print(f"Subset size: {len(articles)} articles")
print(f"Avg article length: {np.mean([len(a.split()) for a in articles]):.0f} words")
print(f"Avg summary length: {np.mean([len(s.split()) for s in summaries]):.0f} words")
print()
print("Sample article (first 300 chars):")
print(articles[0][:300])
print("\nReference summary:")
print(summaries[0])


## §2 – Extractive Summarisation: TextRank

TextRank is a **graph-based** algorithm adapted from PageRank:
1. Represent each sentence as a TF-IDF vector
2. Build a similarity graph where edges = cosine similarity between sentences
3. Run PageRank-style scoring — sentences similar to many others score higher
4. Select the top-k scoring sentences (in original order)

Implemented via the `sumy` library with `LexRankSummarizer` (similar to TextRank).


In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words

LANGUAGE = "english"
SENTENCES_COUNT = 3   # target summary length

stemmer    = Stemmer(LANGUAGE)
summarizer_tr = TextRankSummarizer(stemmer)
summarizer_tr.stop_words = get_stop_words(LANGUAGE)

def textrank_summarize(text, n_sentences=SENTENCES_COUNT):
    parser  = PlaintextParser.from_string(text, Tokenizer(LANGUAGE))
    summary = summarizer_tr(parser.document, n_sentences)
    return " ".join(str(s) for s in summary)

# ── Generate extractive summaries ─────────────────────────────────────────────
print("Generating TextRank summaries (500 articles)...")
extractive_summaries = [textrank_summarize(a) for a in articles]
print("Done.")
print()
print("Example extractive summary (article 0):")
print(extractive_summaries[0])


## §3 – Abstractive Summarisation: BART-large-cnn

`facebook/bart-large-cnn` is a BART model fine-tuned on CNN/DailyMail for
abstractive summarisation. We use it in **inference mode** (no fine-tuning needed
on our subset — it was already trained on this domain).

BART uses a denoising auto-encoder pre-training objective with a transformer
encoder-decoder architecture.


In [ ]:
import torch
from transformers import BartForConditionalGeneration, BartTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("Loading facebook/bart-large-cnn ...")
_tok   = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
_model = BartForConditionalGeneration.from_pretrained(
    "facebook/bart-large-cnn",
    attn_implementation="eager",
).to(DEVICE)
_model.eval()

def bart_summarize(text, max_new_tokens=130, min_length=30):
    truncated = " ".join(text.split()[:900])
    inputs = _tok(truncated, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    with torch.no_grad():
        ids = _model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_length=min_length,
            num_beams=4,
            early_stopping=True,
        )
    return _tok.decode(ids[0], skip_special_tokens=True)

print("Model loaded.")

print("Generating BART abstractive summaries (this may take 20-40 min on CPU, 5-10 min on GPU)...")
abstractive_summaries = []
for i, article in enumerate(articles):
    if i % 50 == 0:
        print(f"  {i}/500 ...")
    try:
        abstractive_summaries.append(bart_summarize(article))
    except Exception as e:
        print(f"  Warning: article {i} skipped ({e})")
        abstractive_summaries.append("")

print(f"Done. Generated {len(abstractive_summaries)} summaries.")
print("\nExample BART summary (article 0):")
print(abstractive_summaries[0])


## §4 – Evaluation

Metrics:
- **ROUGE-1 / ROUGE-2 / ROUGE-L** – n-gram and longest-common-subsequence overlap
- **BLEU** – n-gram precision with brevity penalty
- **METEOR** – precision/recall with synonym and stemming alignment
- **BERTScore** – contextual semantic similarity (F1 over BERT embeddings)


In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
import sacrebleu
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from nltk.tokenize import word_tokenize

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_rouge(preds, refs):
    r1s, r2s, rls = [], [], []
    for p, r in zip(preds, refs):
        s = rouge.score(r, p)
        r1s.append(s["rouge1"].fmeasure)
        r2s.append(s["rouge2"].fmeasure)
        rls.append(s["rougeL"].fmeasure)
    return np.mean(r1s), np.mean(r2s), np.mean(rls)

def compute_bleu(preds, refs):
    return sacrebleu.corpus_bleu(preds, [refs]).score

def compute_meteor(preds, refs):
    scores = []
    for p, r in zip(preds, refs):
        score = nltk_meteor(
            [word_tokenize(r.lower())],
            word_tokenize(p.lower())
        )
        scores.append(score)
    return np.mean(scores)

# ── ROUGE ─────────────────────────────────────────────────────────────────────
print("Computing ROUGE...")
ext_r1, ext_r2, ext_rl = compute_rouge(extractive_summaries, summaries)
abs_r1, abs_r2, abs_rl = compute_rouge(abstractive_summaries, summaries)

# ── BLEU ──────────────────────────────────────────────────────────────────────
print("Computing BLEU...")
ext_bleu = compute_bleu(extractive_summaries, summaries)
abs_bleu = compute_bleu(abstractive_summaries, summaries)

# ── METEOR ────────────────────────────────────────────────────────────────────
print("Computing METEOR...")
ext_meteor = compute_meteor(extractive_summaries, summaries)
abs_meteor = compute_meteor(abstractive_summaries, summaries)

# ── BERTScore ─────────────────────────────────────────────────────────────────
print("Computing BERTScore (may take a few minutes)...")
_, _, ext_bscore = bert_score_fn(extractive_summaries, summaries, lang="en", verbose=False)
_, _, abs_bscore = bert_score_fn(abstractive_summaries, summaries, lang="en", verbose=False)
ext_bs = ext_bscore.mean().item()
abs_bs = abs_bscore.mean().item()

print("\n=== Summarisation Results ===")
results = pd.DataFrame({
    "Method":    ["TextRank (Extractive)", "BART-large-cnn (Abstractive)"],
    "ROUGE-1":   [ext_r1,    abs_r1],
    "ROUGE-2":   [ext_r2,    abs_r2],
    "ROUGE-L":   [ext_rl,    abs_rl],
    "BLEU":      [ext_bleu,  abs_bleu],
    "METEOR":    [ext_meteor,abs_meteor],
    "BERTScore": [ext_bs,    abs_bs],
})
print(results.to_string(index=False))

# ── Bar chart ──────────────────────────────────────────────────────────────────
metrics = ["ROUGE-1","ROUGE-2","ROUGE-L","BLEU","METEOR","BERTScore"]
bleu_scale = 1/100   # BLEU is 0-100, scale to 0-1
ext_vals = [ext_r1, ext_r2, ext_rl, ext_bleu*bleu_scale, ext_meteor, ext_bs]
abs_vals = [abs_r1, abs_r2, abs_rl, abs_bleu*bleu_scale, abs_meteor, abs_bs]
x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - 0.2, ext_vals, 0.4, label="TextRank", color="steelblue")
ax.bar(x + 0.2, abs_vals, 0.4, label="BART",     color="coral")
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("Score (BLEU scaled to 0-1)")
ax.set_title("Summarisation: TextRank vs BART across all metrics")
ax.legend(); plt.tight_layout()
plt.savefig("summ_comparison.png", dpi=150); plt.show()
print("Saved: summ_comparison.png")


## §5 – Qualitative Examples (3 samples)


In [ ]:
rng2 = np.random.default_rng(SEED + 1)
sample_idx = rng2.choice(len(articles), size=3, replace=False)

for rank, idx in enumerate(sample_idx, 1):
    art = articles[idx]
    print(f"{'='*80}")
    print(f"EXAMPLE {rank}  (article index {idx})")
    print(f"{'='*80}")
    print(f"Article (first 400 chars):\n  {art[:400].replace(chr(10),' ')}...")
    print(f"\nReference summary:\n  {summaries[idx]}")
    print(f"\nTextRank (Extractive):\n  {extractive_summaries[idx]}")
    print(f"\nBART (Abstractive):\n  {abstractive_summaries[idx]}")
    print()


## §6 – Discussion: Extractive vs Abstractive

| Aspect | TextRank (Extractive) | BART (Abstractive) |
|---|---|---|
| **Method** | Graph-based sentence selection | Encoder-decoder generation |
| **Fluency** | Inherits source sentence quality | Generates natural prose |
| **Factual consistency** | Exact source text — always faithful | May hallucinate facts |
| **Information coverage** | Limited to salient sentences | Can compress and rephrase |
| **ROUGE score** | High (copies source n-grams) | Lower (paraphrases instead) |
| **BERTScore** | Moderate (no semantic compression) | Higher (captures meaning) |
| **Compute** | CPU, milliseconds per article | GPU, seconds per article |

**Key observations:**
- TextRank achieves higher ROUGE scores because it copies verbatim sentences from the source,
  maximising n-gram overlap with reference summaries that were written by reading the source.
- BART produces more **fluent and concise** summaries that read naturally, at the cost of
  lower ROUGE (due to paraphrasing) but higher BERTScore (semantic similarity preserved).
- BART can sometimes **hallucinate** entities or facts not present in the source —
  a key reliability concern for abstractive systems.
- For news summarisation, BART is preferred in practice; TextRank requires no training
  and is a strong baseline for resource-constrained settings.
